In [ ]:
### RAG (Retrieval-Augmented Generation)
    - retriever retrives relevant documents
    - LLM generate a response informed by both query and retrived documents

In [ ]:
### 1. Utility method for asking Question & pretty print answers

def generate_answer( question ):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
    )

    return response.choices[0].message.content


def pretty_print( answer ):
    wrapped_text = textwrap.fill( answer, width=80 )
    return wrapped_text

In [ ]:
### 3. City-related RAG 만들기

## 3.1 시티 목록 생성
city_names = [
    "New York City", "London", "Paris", "Tokyo", "Berlin", "Shanghai",
    "Los Angeles", "Istanbul", "Sydney", "Seoul", "Saint Petersburg",
    "Mumbai", "Beijing", "Leipzig", "Toronto", "Rio de Janeiro",
    "Mexico City", "Budapest", "Barcelona", "Moscow", "Vancouver",
    "Dublin", "Antananarivo", "Suwon"
]

city_related_to_question = 'Dresden'

if city_related_to_question not in city_names:
  city_names.append(city_related_to_question)


## 3.2 Wikipedia 에서 city 관련 문서 로드
import wikipedia
wikipedia.set_user_agent("my-rag-project (test@example.com)")
reader = WikipediaReader()
documents = reader.load_data( city_names, auto_suggest=False )


## 3.3 QueryEngine 만들기
index = VectorStoreIndex.from_documents( documents )
query_engine = index.as_query_engine()

response = query_engine.query( "What's the arts and culture scene in Berlin?" )
print(textwrap.fill(str(response), 100))


## 3.4 Custom Query Engine class 로 만들기 - retrieve(), generate_response(), query() 인터페이스 제공

from openai import OpenAI
oai_client = OpenAI()

class RAG_from_scratch:
    def retrieve( self, query: str ) -> list:                               # Retrieve relevant text from vector store.
        results = query_engine.query( query )
        return results

    def generate_response( self, query: str, context_str: list ) -> str:      # Generate answer from context.
        completion = oai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            messages=
            [
                {
                    "role": "user",
                    "content":
                        f"We have provided context information below. \n"
                        f"---------------------\n"
                        f"{context_str}"
                        f"\n---------------------\n"
                        f"Given this information, please answer the question: {query}"
                }
            ]
        ).choices[0].message.content
        return completion

    def query( self, query: str ) -> str:
        context_str = self.retrieve( query )
        completion = self.generate_response( query, context_str )
        return completion

rag = RAG_from_scratch()


## 3.5 query by RAG (city qeustion)

city_question = 'Which Korean city has a relationship with Dresden?'
answer = rag.query( city_question )
print( pretty_print(answer) )


## 3.6 check retriver results

retriever = index.as_retriever()

ret = retriever.retrieve( city_question )
for i in range(len(ret)):
  print(f"Retrieved Paasge {i+1}\n", ret[i].text, '\n\n\n')

In [ ]:
### 4. custom retriver, custom query engine 을 이용한 RAG 구성
    - 고려사항
        - embedding (tokenizer)
        - LLM : temperature, top-p
        - parser : chunk size, chunk overwrap
        - retriver similarity top k : 1, 2, 3 ...
        - llm qa prompt : answer, summarize

## 4.1 embedding
from llama_index.core import Settings

Settings.embed_model = OpenAIEmbedding()            # Option 1. global default
Settings.embed_model = HuggingFaceEmbedding(        # Option 2. local model
    model_name="BAAI/bge-small-en-v1.5"                 # change this to the local path or huggingface model name
)

#per-index
index = VectorStoreIndex.from_documents( documents, embed_model=embed_model )


## 4.2 LLM options
response = oai_client.chat.completions.create(
    model="gpt-3.5-turbo",                          # model
    temperature=0,                                  # T
    top_p=0,                                        # top p
    messages=messages,
)


## 4.3 parser & retriver similarity top k
text_splitter_short = SentenceSplitter( chunk_size=200, chunk_overlap=50 )
index_short = VectorStoreIndex.from_documents( documents=documents, transformations=[text_splitter_short] )

text_splitter_long = SentenceSplitter( chunk_size=1024, chunk_overlap=200 )
index_long = VectorStoreIndex.from_documents( documents=documents, transformations=[text_splitter_long] )

retriever_short = index_short.as_retriever( similarity_top_k=1 )
retriever_long = index_long.as_retriever( similarity_top_k=1 )


## 4.4 qa prompt

simple_qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

short_sum_prompt = PromptTemplate(
    """Write a summary of the following. Try to use only the information provided.
Try to include as many key details as possible.
---------------------\n
{context_str}
---------------------\n
SUMMARY:"""
)


retriever = index_short.as_retriever( similarity_top_k=2 )
index = index_long


In [ ]:
### 최종 Refine RAG

from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer
from llama_index.llms.openai import OpenAI
from llama_index.core import PromptTemplate

simple_qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

short_sum_prompt = PromptTemplate(
"""Write a summary of the following. Try to use only the information provided.
Try to include as many key details as possible.
---------------------\n
{context_str}
---------------------\n
SUMMARY:"""
)


# =============================
class OurCustomQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OpenAI
    qa_prompt: PromptTemplate = simple_qa_prompt

    def custom_query( self, query_str: str ):
        nodes = self.retriever.retrieve( query_str )
        context_str = "\n\n".join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            self.qa_prompt.format( context_str=context_str, query_str=query_str )
        )

        return str(response)


# =============================
llm = OpenAI(model="gpt-3.5-turbo")
retriever = index.as_retriever()
synthesizer = get_response_synthesizer(response_mode="compact")

query_engine_answer = OurCustomQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    llm=llm,
    qa_prompt=simple_qa_prompt,
)

query_engine_sum = OurCustomQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    llm=llm,
    qa_prompt=short_sum_prompt,
)

res_answer = query_engine_answer.custom_query("What's the arts and culture scene in Berlin?")
res_summary = query_engine_sum.custom_query("What's the arts and culture scene in Berlin?")


# =============================
class Refine_RAG:
    def retrieve( self, query: str ) -> list:
        ret = retriever.retrieve( query )
        results = query_engine.query( query )
        return ret, results

    def generate_response( self, query: str, context_str: list ) -> str:        # Generate answer from context.
        messages = [
            {
                "role": "system",
                "content": f"You are a helpful assistant. Answer as concisely as possible.",
            },
            {
                "role": "user",
                "content":
                    f"""
                    ###Instruction
Please answer the following question based on the provided context. Your answer should be short and concise.
Basically, you have to answer the question based on the provided context. But you can use your parametrized knowledge when the provided context was wrong or unrelated to the quesion.
When you generate the answer, you should explain the reason why you deduced such a answer from the context.

You should use below format:

### Answer : <your answer>
### Reason : <your reason>

### Question
{query}

### Provided Context
{context_str}
                    """
            }
        ]

        response = oai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            messages=messages,
        )

        return response.choices[0].message.content

    def query( self, query: str ) -> str:
        ret, context_str = self.retrieve(query)
        # for i in range(len(ret)):
        #   print("Retrieved Context: \n", ret[i].text) # use only when you want to see intermeidate result
        # print("\n\nIntermediate Summary: \n",  context_str) # use only when you want to see intermeidate result
        completion = self.generate_response( query, context_str )
        return completion, ret

refine_rag = Refine_RAG()


sample_question = "City council of Suwon addressed illegal dumping of household waste in what way?"
answer, _ = refine_rag.query(sample_question)
print(f"\n\n{answer}")